# 03 Hypotheses H1, H3, H4

Definitions: `docs/delta_lake_handoff/DECISIONS.md`. Computed by `python -m etl.hypotheses` from Gold (versions below). Intervals: Poisson bootstrap over pages, B = 2,000, seed 226.

In [1]:
import json, os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from deltalake import DeltaTable
LAKE = REPO / 'data' / 'lake'
RUNS = LAKE / '_runs'
def run(name): return json.loads((RUNS / name).read_text())
def table(name, version=None, columns=None):
    return DeltaTable(str(LAKE / name), version=version).to_pandas(columns=columns)
def history(name, n=10):
    h = DeltaTable(str(LAKE / name)).history(n)
    return pd.DataFrame([{k: e.get(k) for k in ('version', 'operation', 'step', 'as_of', 'snapshot', 'rows')} for e in h])
h = run('hypotheses.json')
print('Gold versions read:', h['gold_versions'])

Gold versions read: {'fact_edit': 3, 'dim_page': 3, 'fact_revert_label': 1, 'mart_el_curve': 2}


## H1: share of eventual reverts known at 1 h (threshold 60%) and 48 h (threshold 95%)

In [2]:
pd.DataFrame(h['H1'])[['series', 'eventual_reverts', 'pages', 'known_1h_pct', 'known_1h_ci', 'known_48h_pct', 'known_48h_ci', 'verdict']]

,series,eventual_reverts,pages,known_1h_pct,known_1h_ci,known_48h_pct,known_48h_ci,verdict
0,history,17646,10364,55.66,"[54.49, 56.88]",94.78,"[94.12, 95.4]",not falsified; an interval crosses its thresho...
1,live,17818,10471,57.95,"[56.71, 59.28]",93.72,"[92.66, 94.59]",not falsified
2,live_ext,112937,56531,58.74,"[58.04, 59.49]",95.07,"[94.72, 95.37]",falsified (point estimate); an interval crosse...
3,live_ext_october,67998,35420,58.11,"[57.29, 59.0]",94.64,"[94.1, 95.08]",not falsified; an interval crosses its thresho...


## H3: E(L) for burst edits (>= 10 page edits in the prior hour) against the rest

In [3]:
pd.DataFrame(h['H3'])[['series', 'L', 'burst_reverts', 'E_burst', 'E_burst_ci', 'E_rest', 'E_rest_ci', 'diff_burst_minus_rest', 'diff_ci', 'verdict']]

,series,L,burst_reverts,E_burst,E_burst_ci,E_rest,E_rest_ci,diff_burst_minus_rest,diff_ci,verdict
0,history,1 h,343,0.2566,"[0.1367, 0.4051]",0.4471,"[0.4353, 0.4587]",-0.1905,"[-0.3125, -0.0532]","falsified: burst E(L) lower, intervals apart"
1,history,1 d,343,0.0700,"[0.006, 0.1566]",0.1113,"[0.103, 0.12]",-0.0413,"[-0.105, 0.0509]",falsified (intervals overlap)
2,live,1 h,439,0.2255,"[0.0479, 0.4503]",0.4254,"[0.4138, 0.4369]",-0.1999,"[-0.3716, 0.0097]",falsified (intervals overlap)
3,live,1 d,439,0.1367,"[0.0023, 0.3748]",0.1048,"[0.097, 0.1131]",0.0318,"[-0.1033, 0.2528]",falsified (intervals overlap)
4,live_ext,1 h,3401,0.2887,"[0.1884, 0.3965]",0.4164,"[0.4111, 0.4218]",-0.1277,"[-0.2259, -0.021]","falsified: burst E(L) lower, intervals apart"
5,live_ext,1 d,3401,0.0641,"[0.012, 0.1366]",0.0960,"[0.0929, 0.0991]",-0.0319,"[-0.0834, 0.0475]",falsified (intervals overlap)
6,live_ext_october,1 h,2038,0.3548,"[0.2122, 0.4923]",0.4208,"[0.4146, 0.4268]",-0.0661,"[-0.2122, 0.0704]",falsified (intervals overlap)
7,live_ext_october,1 d,2038,0.1021,"[0.0168, 0.2165]",0.1003,"[0.0961, 0.1046]",0.0018,"[-0.0836, 0.1115]",falsified (intervals overlap)


## H4: title at event time (dim_page version) against the current title

In [4]:
pd.DataFrame({k: v for k, v in h['H4'].items() if isinstance(v, dict)})

,history,live_ext
n,295016,2397738
moved,3381,16762
pages,165415,1238130
pages_moved,966,3454
titles,166059,1238578
titles_changed,1853,6779
titles_rate_changed,1497,3154
titles_reverts_changed,168,601
reverts_moved,406,1596
share_facts_title_differs_pct,1.146,0.699


In [5]:
print('history_revisions rows with page_title != page_title_historical:', h['H4']['history_revisions_title_differs_from_historical'])

history_revisions rows with page_title != page_title_historical: 2319


## E(L) by burst group (gold/mart_el_by_burst)

In [6]:
b = table('gold/mart_el_by_burst')
b[b.wait_label.isin(['1 h', '1 d'])].pivot_table(index=['cohort_id', 'label_source', 'burst_group'], columns='wait_label', values=['n_eventual', 'e_l']).round(4)

e_l         n_eventual          
wait_label                             1 d     1 h        1 d       1 h
cohort_id label_source burst_group                                     
history   dump         0            0.1132  0.4619    11390.0   11390.0
                       1            0.1354  0.5062     2519.0    2519.0
                       10+          0.0700  0.2566      343.0     343.0
                       2-4          0.0883  0.3628     2627.0    2627.0
                       5-9          0.0821  0.3220      767.0     767.0
                       all          0.1104  0.4434    17646.0   17646.0
live      dump         0            0.1094  0.4348    11634.0   11634.0
                       1            0.1437  0.5126     2499.0    2499.0
                       10+          0.1611  0.2472      453.0     453.0
                       2-4          0.0874  0.3588     2667.0    2667.0
                       5-9          0.1347  0.3689      824.0     824.0
                       all          0.1133  0.4266    18077.0   18077.0
          stream       0            0.1059  0.4311    11521.0   11521.0
                       1            0.1343  0.5053     2464.0    2464.0
                       10+          0.1367  0.2255      439.0     439.0
                       2-4          0.0747  0.3499     2612.0    2612.0
                       5-9          0.0972  0.3414      782.0     782.0
                       all          0.1056  0.4205    17818.0   17818.0
live_ext  stream       0            0.0996  0.4292    72256.0   72256.0
                       1            0.1135  0.4780    15281.0   15281.0
                       10+          0.0641  0.2887     3401.0    3401.0
                       2-4          0.0737  0.3363    16407.0   16407.0
                       5-9          0.0658  0.3180     5592.0    5592.0
                       all          0.0950  0.4126   112937.0  112937.0